# 실습 4. 쿼리 변환 — Multi-Query · HyDE

## 실습 목표

질문과 문서의 **어휘가 다르면**(짧거나 모호한 질의) 벡터 검색도 정답을 놓칩니다(교안 4.4~4.6).
질문 자체를 바꿔 **정답 문서의 순위를 끌어올립니다.**

- **Multi-Query**: LLM으로 질문을 여러 표현으로 늘려 각각 검색 → RRF 융합
- **HyDE**: 질문에 대한 '가상 답변'을 만들어 그 문장으로 검색(문서와 어휘가 비슷해져 적중↑)
- **target(정답) 기반**: baseline이 놓친 정답을 변환이 몇 위까지 끌어올리는지 비교

> LLM 호출이 있으므로 루트 `.env` 의 `MLAPI_*` 가 필요합니다.

## 1. 준비 — 검색기 + LLM + 변환 헬퍼

짧고 모호한 질의 **"한 번 더 거르기"** 를 씁니다. 정답은 doc04(리랭킹)인데, 질의에 'reranking·후보·
정렬' 같은 단어가 없어 **baseline 벡터가 정답을 한참 아래로** 보냅니다(어휘 격차).

In [1]:
from _common import (rag_tech_documents, build_vector_retriever,
                     get_llm, multi_query, hyde, rrf_fuse, rank_of)

QUERY, TARGET = "한 번 더 거르기", "doc04"   # 정답 doc04(리랭킹) — 질의와 어휘가 거의 안 겹침
docs = rag_tech_documents()
vec = build_vector_retriever(docs, k=10, name="d21_qt_nb")
llm = get_llm()

def topics(ds): return [d.metadata.get("topic", "?") for d in ds]

base_rank = rank_of(TARGET, vec.invoke(QUERY))
print(f"질의={QUERY!r} · 정답={TARGET}")
print(f"baseline(원 질문 그대로) 정답 순위: {base_rank}  ← 어휘 격차로 한참 아래")

/Users/bkk/프로젝트/yeardream/.venv-1/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6628.49it/s]


질의='한 번 더 거르기' · 정답=doc04
baseline(원 질문 그대로) 정답 순위: 7  ← 어휘 격차로 한참 아래


## 2. Multi-Query — 여러 표현으로 늘려 검색

`multi_query(llm, q, n)` 는 LLM으로 질문을 n개 변형으로 늘립니다(원 질문 포함).
각 변형을 검색해 `rrf_fuse` 로 합치면, 어휘가 보강돼 **정답 순위가 올라갑니다.**

In [ ]:
variants = multi_query(llm, QUERY, n=3)
print("생성된 변형 질의:")
for v in variants:      #원래 표현에 증강한 표현이 더해져 1+n개 만들어짐!
    print("   -", v)

fused = rrf_fuse([vec.invoke(v) for v in variants], top_n=10)
mq_rank = rank_of(TARGET, fused)
print(f"\n[Multi-Query] 정답 순위: {base_rank} → {mq_rank}  (융합 top: {topics(fused)[:5]})")

생성된 변형 질의:
   - 한 번 더 거르기
   - 한 번 더 거르기 뜻
   - 한 번 더 거르기 의미
   - 한 번 더 거르다 표현 설명

[Multi-Query] 정답 순위: 7 → 8  (융합 top: ['쿼리변환', 'BM25', 'Cohere리랭크', '하이브리드', '벡터검색'])


## 3. HyDE — 가상 답변으로 검색

질문에 대한 가상 답변을 만들면, 그 답변에 'reranking·후보·재정렬' 같은 **정답 문서의 어휘**가
들어가 검색이 정답에 가까워집니다.

In [ ]:
h = hyde(llm, QUERY)
print("가상 답변:", h[:100], "...")
hyde_rank = rank_of(TARGET, vec.invoke(h))
print(f"\n[HyDE] 정답 순위: {base_rank} → {hyde_rank}")
print(f"\n요약 — 정답 doc04 순위:  baseline={base_rank}  MultiQuery={mq_rank}  HyDE={hyde_rank}")

**포인트**

- baseline은 어휘 격차로 정답(doc04)을 **한참 아래**(예: 7위)로 보냄
- **Multi-Query**·**HyDE** 는 질의 어휘를 보강해 정답을 **위로 끌어올림**(예: 5위·3위)
- 단, **HyDE** 는 질문이 모호하면 엉뚱한 도메인으로 샐 수도 있음 → 도메인을 명시하거나 Multi-Query 병행
- (LLM 비결정성으로 변형·순위는 실행마다 약간 달라질 수 있음)

# [TODO]

## [TODO] 1. 변형 질의 개수를 늘려보기

`multi_query(llm, QUERY, n=5)` 로 변형을 **5개** 생성해 융합하고, 정답 doc04 순위가
n=3 때와 어떻게 달라지는지 출력하세요.

In [ ]:
# [TODO] 1: n=5 변형 융합 후 정답 순위 출력
# [TODO] 여기에 코드를 작성하세요.
variants5 = multi_query(llm, QUERY, n=5)

print("생성된 n=5 변형 질의:")
for v in variants5:
    print("   -", v)

fused5 = rrf_fuse([vec.invoke(v) for v in variants5], top_n=10)
mq5_rank = rank_of(TARGET, fused5)
print(f"\n[Multi-Query n=5] 정답 순위: n=3 {mq_rank} → n=5 {mq5_rank}  (융합 top: {topics(fused5)[:5]})")


## [TODO] 2. 도메인을 명시한 HyDE

HyDE의 약점(도메인 오해)을 줄이기 위해 **질문에 도메인 힌트를 덧붙여** 가상 답변을 만들고,
정답 순위가 어떻게 달라지는지 원본 HyDE와 비교하세요.
- `QUERY_HINT = QUERY + " (RAG 검색·리랭킹 맥락에서)"`

In [ ]:
# [TODO] 2: 도메인 힌트 HyDE 와 원본 HyDE 정답 순위 비교
# [TODO] 여기에 코드를 작성하세요.
QUERY_HINT = QUERY + " (RAG 검색·리랭킹 맥락에서)"

h_hint = hyde(llm, QUERY_HINT)
print("도메인 힌트 HyDE 가상 답변:", h_hint[:100], "...")

hint_rank = rank_of(TARGET, vec.invoke(h_hint))
print(f"\n[HyDE + 도메인 힌트] 정답 순위: 원본 HyDE {hyde_rank} → 힌트 HyDE {hint_rank}")
print(f"요약 — baseline={base_rank}  HyDE={hyde_rank}  HyDE+hint={hint_rank}")


## 실습 정리

- 짧고 모호한 질의의 **어휘 격차**를 Multi-Query / HyDE 로 메워 **정답 순위를 끌어올림**
- HyDE의 도메인 오해 함정과 완화법(도메인 명시) 확인
- 모든 변환은 **추가 LLM 호출 비용** → 검색이 잘 안 될 때 선택적으로